In [16]:
import os

# 1. Define the full path to New_dataset.zip in Google Drive
zip_file_path = '/content/drive/MyDrive/New_dataset.zip'

# 2. Create a new local directory for extraction
extraction_dir = './extracted_dataset'
if not os.path.exists(extraction_dir):
    os.makedirs(extraction_dir)
    print(f"Created extraction directory: {extraction_dir}")
else:
    print(f"Extraction directory already exists: {extraction_dir}")

# 3. Use the unzip command to extract the contents
#    -o flag for overwrite without prompt
#    -d flag to specify the destination directory
print(f"\nExtracting '{zip_file_path}' to '{extraction_dir}'...")
!unzip -o "{zip_file_path}" -d "{extraction_dir}"

print("\nExtraction complete. Verifying contents:")
# 4. After extraction, list the contents of the extracted_dataset directory recursively
!ls -R "{extraction_dir}"

Streaming output truncated to the last 5000 lines.
test_0172_aligned.jpg  test_1935_aligned.jpg  test_2344_aligned.jpg
test_0263_aligned.jpg  test_1954_aligned.jpg  test_2345_aligned.jpg
test_0280_aligned.jpg  test_1955_aligned.jpg  test_2346_aligned.jpg
test_0281_aligned.jpg  test_1958_aligned.jpg  test_2347_aligned.jpg
test_0283_aligned.jpg  test_1963_aligned.jpg  test_2348_aligned.jpg
test_0343_aligned.jpg  test_1988_aligned.jpg  test_2349_aligned.jpg
test_0357_aligned.jpg  test_1990_aligned.jpg  test_2350_aligned.jpg
test_0420_aligned.jpg  test_1993_aligned.jpg  test_2351_aligned.jpg
test_0432_aligned.jpg  test_2034_aligned.jpg  test_2352_aligned.jpg
test_0639_aligned.jpg  test_2037_aligned.jpg  test_2353_aligned.jpg
test_0640_aligned.jpg  test_2154_aligned.jpg  test_2354_aligned.jpg
test_0734_aligned.jpg  test_2159_aligned.jpg  test_2355_aligned.jpg
test_0774_aligned.jpg  test_2213_aligned.jpg  test_2356_aligned.jpg
test_0804_aligned.jpg  test_2214_aligned.jpg  test_2357_aligned.j

In [17]:
# First, list the contents of the main extraction directory to verify 'DATASET'
print("Contents of 'extracted_dataset':")
!ls -F extracted_dataset

# Then, try listing the contents of the DATASET directory if it exists
print("\nAttempting to list 'extracted_dataset/DATASET':")
!ls -F extracted_dataset/DATASET

Contents of 'extracted_dataset':
DATASET/  test_labels.csv  train_labels.csv

Attempting to list 'extracted_dataset/DATASET':
test/  train/


In [18]:
import pandas as pd

# Define paths to the label files
train_labels_path = 'extracted_dataset/train_labels.csv'
test_labels_path = 'extracted_dataset/test_labels.csv'

# Load the label files into pandas DataFrames
train_df = pd.read_csv(train_labels_path)
test_df = pd.read_csv(test_labels_path)

print("Train Labels DataFrame Head:")
print(train_df.head())

print("\nTest Labels DataFrame Head:")
print(test_df.head())

# Also check the column names and data types to understand the structure
print("\nTrain DataFrame Info:")
train_df.info()

print("\nTest DataFrame Info:")
test_df.info()

Train Labels DataFrame Head:
                     image  label
0  train_00001_aligned.jpg      5
1  train_00002_aligned.jpg      5
2  train_00003_aligned.jpg      4
3  train_00004_aligned.jpg      4
4  train_00005_aligned.jpg      5

Test Labels DataFrame Head:
                   image  label
0  test_0001_aligned.jpg      5
1  test_0002_aligned.jpg      1
2  test_0003_aligned.jpg      4
3  test_0004_aligned.jpg      1
4  test_0005_aligned.jpg      5

Train DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12271 entries, 0 to 12270
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   image   12271 non-null  object
 1   label   12271 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 191.9+ KB

Test DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3068 entries, 0 to 3067
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   image   3068 n

In [19]:
# Define the mapping from numerical labels to emotion names for RAF-DB dataset
# Common mapping for RAF-DB:
# 1: Surprise, 2: Fear, 3: Disgust, 4: Happiness, 5: Sadness, 6: Anger, 7: Neutral

emotion_map = {
    1: 'Surprise',
    2: 'Fear',
    3: 'Disgust',
    4: 'Happiness',
    5: 'Sadness',
    6: 'Anger',
    7: 'Neutral'
}

# Apply the mapping to the 'label' column for both train and test DataFrames
train_df['emotion'] = train_df['label'].map(emotion_map)
test_df['emotion'] = test_df['label'].map(emotion_map)

print("\nTrain Labels DataFrame with Emotion Names:")
print(train_df.head())

print("\nTest Labels DataFrame with Emotion Names:")
print(test_df.head())

# Display the value counts for the new 'emotion' column to verify
print("\nTrain Emotion Distribution:")
print(train_df['emotion'].value_counts())

print("\nTest Emotion Distribution:")
print(test_df['emotion'].value_counts())


Train Labels DataFrame with Emotion Names:
                     image  label    emotion
0  train_00001_aligned.jpg      5    Sadness
1  train_00002_aligned.jpg      5    Sadness
2  train_00003_aligned.jpg      4  Happiness
3  train_00004_aligned.jpg      4  Happiness
4  train_00005_aligned.jpg      5    Sadness

Test Labels DataFrame with Emotion Names:
                   image  label    emotion
0  test_0001_aligned.jpg      5    Sadness
1  test_0002_aligned.jpg      1   Surprise
2  test_0003_aligned.jpg      4  Happiness
3  test_0004_aligned.jpg      1   Surprise
4  test_0005_aligned.jpg      5    Sadness

Train Emotion Distribution:
emotion
Happiness    4772
Neutral      2524
Sadness      1982
Surprise     1290
Disgust       717
Anger         705
Fear          281
Name: count, dtype: int64

Test Emotion Distribution:
emotion
Happiness    1185
Neutral       680
Sadness       478
Surprise      329
Anger         162
Disgust       160
Fear           74
Name: count, dtype: int64


### 1. Construct Full Image Paths

The image filenames in the `train_df` and `test_df` are relative. We need to create absolute paths to these images on disk. Given that your `extracted_dataset` directory contains `DATASET/train` and `DATASET/test` folders, we'll combine these paths to form the full image locations.

In [20]:
import os

# Define the base directory for images
base_image_dir = 'extracted_dataset/DATASET'

# Create full paths for training images, including the label as a subdirectory
train_df['path'] = base_image_dir + '/train/' + train_df['label'].astype(str) + '/' + train_df['image']

# Create full paths for testing images, including the label as a subdirectory
test_df['path'] = base_image_dir + '/test/' + test_df['label'].astype(str) + '/' + test_df['image']

print("Train DataFrame with full image paths:")
print(train_df.head())

print("\nTest DataFrame with full image paths:")
print(test_df.head())

Train DataFrame with full image paths:
                     image  label    emotion  \
0  train_00001_aligned.jpg      5    Sadness   
1  train_00002_aligned.jpg      5    Sadness   
2  train_00003_aligned.jpg      4  Happiness   
3  train_00004_aligned.jpg      4  Happiness   
4  train_00005_aligned.jpg      5    Sadness   

                                                path  
0  extracted_dataset/DATASET/train/5/train_00001_...  
1  extracted_dataset/DATASET/train/5/train_00002_...  
2  extracted_dataset/DATASET/train/4/train_00003_...  
3  extracted_dataset/DATASET/train/4/train_00004_...  
4  extracted_dataset/DATASET/train/5/train_00005_...  

Test DataFrame with full image paths:
                   image  label    emotion  \
0  test_0001_aligned.jpg      5    Sadness   
1  test_0002_aligned.jpg      1   Surprise   
2  test_0003_aligned.jpg      4  Happiness   
3  test_0004_aligned.jpg      1   Surprise   
4  test_0005_aligned.jpg      5    Sadness   

                          

### 2. Verify Image Existence

It's important to ensure that all image paths in the DataFrames actually point to existing files. If some files are missing, it can cause errors during model training. We'll check for this and remove any entries corresponding to non-existent files.

In [21]:
# Function to check if a file exists
def check_file_exists(file_path):
    return os.path.exists(file_path)

# Apply the check to both DataFrames
train_df['file_exists'] = train_df['path'].apply(check_file_exists)
test_df['file_exists'] = test_df['path'].apply(check_file_exists)

# Filter out rows where the file does not exist
train_df_cleaned = train_df[train_df['file_exists']].drop(columns=['file_exists'])
test_df_cleaned = test_df[test_df['file_exists']].drop(columns=['file_exists'])

print(f"Original train_df size: {len(train_df)}, Cleaned train_df size: {len(train_df_cleaned)}")
print(f"Original test_df size: {len(test_df)}, Cleaned test_df size: {len(test_df_cleaned)}")

# Overwrite original dataframes with cleaned ones
train_df = train_df_cleaned
test_df = test_df_cleaned

print("\nTrain DataFrame after checking for existing image files:")
print(train_df.head())
print("\nTest DataFrame after checking for existing image files:")
print(test_df.head())

Original train_df size: 12271, Cleaned train_df size: 12271
Original test_df size: 3068, Cleaned test_df size: 3068

Train DataFrame after checking for existing image files:
                     image  label    emotion  \
0  train_00001_aligned.jpg      5    Sadness   
1  train_00002_aligned.jpg      5    Sadness   
2  train_00003_aligned.jpg      4  Happiness   
3  train_00004_aligned.jpg      4  Happiness   
4  train_00005_aligned.jpg      5    Sadness   

                                                path  
0  extracted_dataset/DATASET/train/5/train_00001_...  
1  extracted_dataset/DATASET/train/5/train_00002_...  
2  extracted_dataset/DATASET/train/4/train_00003_...  
3  extracted_dataset/DATASET/train/4/train_00004_...  
4  extracted_dataset/DATASET/train/5/train_00005_...  

Test DataFrame after checking for existing image files:
                   image  label    emotion  \
0  test_0001_aligned.jpg      5    Sadness   
1  test_0002_aligned.jpg      1   Surprise   
2  test_0003

### 3. Encode Categorical Labels

Machine learning models typically require numerical input. Our `emotion` column contains string labels (e.g., 'Happiness', 'Sadness'). We need to convert these into integers. We'll use `LabelEncoder` from `sklearn.preprocessing` for this, ensuring consistent encoding across both training and testing sets.

In [22]:
from sklearn.preprocessing import LabelEncoder

# Initialize LabelEncoder
label_encoder = LabelEncoder()

# Fit on the combined emotion labels from both train and test to ensure consistency
# This is important so that the same emotion always maps to the same integer
all_emotions = pd.concat([train_df['emotion'], test_df['emotion']], axis=0).unique()
label_encoder.fit(all_emotions)

# Transform the 'emotion' column in both DataFrames
train_df['encoded_label'] = label_encoder.transform(train_df['emotion'])
test_df['encoded_label'] = label_encoder.transform(test_df['emotion'])

# Display the mapping
print("Emotion to Encoded Label Mapping:")
for i, emotion in enumerate(label_encoder.classes_):
    print(f"{emotion}: {i}")

print("\nTrain DataFrame with encoded labels:")
print(train_df.head())

print("\nTest DataFrame with encoded labels:")
print(test_df.head())

Emotion to Encoded Label Mapping:
Anger: 0
Disgust: 1
Fear: 2
Happiness: 3
Neutral: 4
Sadness: 5
Surprise: 6

Train DataFrame with encoded labels:
                     image  label    emotion  \
0  train_00001_aligned.jpg      5    Sadness   
1  train_00002_aligned.jpg      5    Sadness   
2  train_00003_aligned.jpg      4  Happiness   
3  train_00004_aligned.jpg      4  Happiness   
4  train_00005_aligned.jpg      5    Sadness   

                                                path  encoded_label  
0  extracted_dataset/DATASET/train/5/train_00001_...              5  
1  extracted_dataset/DATASET/train/5/train_00002_...              5  
2  extracted_dataset/DATASET/train/4/train_00003_...              3  
3  extracted_dataset/DATASET/train/4/train_00004_...              3  
4  extracted_dataset/DATASET/train/5/train_00005_...              5  

Test DataFrame with encoded labels:
                   image  label    emotion  \
0  test_0001_aligned.jpg      5    Sadness   
1  test_0002_al

### Next Steps for Data Preparation: Image Preprocessing and Data Loaders

Now that we have clean DataFrames with full image paths and encoded labels, the next crucial steps for preparing the data for training involve handling the image data itself:

1.  **Image Loading and Decoding:** Reading the image files from their paths.
2.  **Resizing:** Ensuring all images have a consistent size for model input.
3.  **Normalization:** Scaling pixel values (e.g., to a range of 0-1 or -1 to 1).
4.  **Data Augmentation:** Applying random transformations (e.g., rotations, flips, zooms) to the training images to increase dataset variability and improve model generalization.
5.  **Creating Data Loaders:** Building efficient data pipelines (e.g., using TensorFlow's `tf.data` or PyTorch's `DataLoader`) that can load, preprocess, and batch images and their corresponding labels for training and validation.

These steps are typically implemented within a custom dataset class or data generator to handle the images efficiently during model training.

### 4. Image Preprocessing and Data Loaders

To prepare the images for our model, we'll implement a custom dataset and data loaders. This will handle:

*   **Loading images** from the paths in our DataFrames.
*   **Applying transformations** (resizing, normalization, data augmentation).
*   **Batching** the data for efficient training.

We'll define standard image sizes and augmentations for the training set to improve model generalization, while the validation/test set will only undergo resizing and normalization.

In [23]:
import torch
from torchvision import transforms
from PIL import Image
from torch.utils.data import Dataset, DataLoader

# Define image transformations
# We'll use a common input size for models, e.g., 224x224 pixels
IMAGE_SIZE = 224

# Training transformations (with data augmentation)
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE), # Randomly crop and resize to IMAGE_SIZE
    transforms.RandomHorizontalFlip(),       # Randomly flip images horizontally
    transforms.RandomRotation(10),           # Randomly rotate images by +/- 10 degrees
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1), # Randomly change brightness, contrast, saturation, and hue
    transforms.ToTensor(),                   # Convert PIL Image to PyTorch Tensor (scales to 0-1)
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # Normalize with ImageNet stats
])

# Validation/Test transformations (no data augmentation)
test_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), # Resize to IMAGE_SIZE x IMAGE_SIZE
    transforms.ToTensor(),                     # Convert PIL Image to PyTorch Tensor (scales to 0-1)
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # Normalize with ImageNet stats
])

print("Image transformations defined.")

Image transformations defined.


### Custom Dataset Class

We'll create a custom PyTorch `Dataset` class to handle loading the images and applying the defined transformations. This class will take our DataFrames as input.

In [24]:
class EmotionDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_path = self.dataframe.iloc[idx]['path']
        image = Image.open(img_path).convert('RGB') # Ensure image is RGB
        label = self.dataframe.iloc[idx]['encoded_label']

        if self.transform:
            image = self.transform(image)

        return image, label

print("Custom EmotionDataset class defined.")

Custom EmotionDataset class defined.


### 6. Model Training

We will now implement the training loop. This involves iterating over the training data, performing forward and backward passes, and updating the model's weights using the optimizer. We will also track the training loss and accuracy.

In [27]:
import time

# Function to train the model
def train_model(model, train_loader, criterion, optimizer, num_epochs=10):
    model.train() # Set the model to training mode
    start_time = time.time()

    for epoch in range(num_epochs):
        running_loss = 0.0
        correct_predictions = 0
        total_samples = 0

        for i, (inputs, labels) in enumerate(train_loader):
            inputs, labels = inputs.to(device), labels.to(device)

            # Zero the parameter gradients
            optimizer.zero_grad()

            # Forward pass
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            # Backward pass and optimize
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)

            _, predicted = torch.max(outputs.data, 1)
            total_samples += labels.size(0)
            correct_predictions += (predicted == labels).sum().item()

        epoch_loss = running_loss / total_samples
        epoch_accuracy = correct_predictions / total_samples
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}, Accuracy: {epoch_accuracy:.4f}')

    end_time = time.time()
    print(f'Finished Training. Total time: {end_time - start_time:.2f} seconds')

# ### 7. Model Evaluation
#
# After training, we need to evaluate the model's performance on the unseen test dataset.

def evaluate_model(model, test_loader):
    model.eval() # Set the model to evaluation mode
    correct_predictions = 0
    total_samples = 0
    all_predictions = []
    all_true_labels = []

    with torch.no_grad(): # Disable gradient calculation during evaluation
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)

            total_samples += labels.size(0)
            correct_predictions += (predicted == labels).sum().item()

            all_predictions.extend(predicted.cpu().numpy())
            all_true_labels.extend(labels.cpu().numpy())

    accuracy = correct_predictions / total_samples
    print(f'Accuracy on the test set: {accuracy:.4f}')
    return accuracy, all_predictions, all_true_labels

print("Training and evaluation functions defined.")

Training and evaluation functions defined.


### 8. Run Training and Evaluation

Now, let's execute the training and evaluation steps. We'll start with a small number of epochs. You can adjust `num_epochs` to train for longer if needed.

In [32]:
from sklearn.metrics import classification_report
import os

# Set the number of epochs
NUM_EPOCHS = 20 # Increased epochs as requested

print(f"Starting training for {NUM_EPOCHS} epochs...")

# Train the model
train_model(model, train_loader, criterion, optimizer, num_epochs=NUM_EPOCHS)

print("\nEvaluating model on the test set...")

# Evaluate the model
test_accuracy, all_predictions, all_true_labels = evaluate_model(model, test_loader)

print(f"Final Test Accuracy: {test_accuracy:.4f}")

# --- Save Results to Google Drive ---

# Define a directory in Google Drive to save results
save_dir = '/content/drive/MyDrive/emotion_model_results'
os.makedirs(save_dir, exist_ok=True)
print(f"\nSaving results to: {save_dir}")

# 1. Save the trained model's state dictionary
model_save_path = os.path.join(save_dir, 'resnet18_emotion_model.pth')
torch.save(model.state_dict(), model_save_path)
print(f"Trained model saved to {model_save_path}")

# 2. Save Evaluation Metrics and Classification Report
metrics_report_path = os.path.join(save_dir, 'evaluation_report.txt')
with open(metrics_report_path, 'w') as f:
    f.write(f"Final Test Accuracy: {test_accuracy:.4f}\n\n")

    # Generate and write classification report
    target_names = [label_encoder.inverse_transform([i])[0] for i in range(num_classes)]
    report = classification_report(all_true_labels, all_predictions, target_names=target_names)
    f.write("Classification Report:\n")
    f.write(report)
print(f"Evaluation metrics and classification report saved to {metrics_report_path}")


Starting training for 20 epochs...
Epoch [1/20], Loss: 1.4466, Accuracy: 0.4558
Epoch [2/20], Loss: 1.2511, Accuracy: 0.5418
Epoch [3/20], Loss: 1.1474, Accuracy: 0.5837
Epoch [4/20], Loss: 1.0834, Accuracy: 0.6088
Epoch [5/20], Loss: 1.0273, Accuracy: 0.6292
Epoch [6/20], Loss: 1.0045, Accuracy: 0.6428
Epoch [7/20], Loss: 0.9715, Accuracy: 0.6504
Epoch [8/20], Loss: 0.9564, Accuracy: 0.6587
Epoch [9/20], Loss: 0.9308, Accuracy: 0.6648
Epoch [10/20], Loss: 0.9033, Accuracy: 0.6741
Epoch [11/20], Loss: 0.8801, Accuracy: 0.6859
Epoch [12/20], Loss: 0.8668, Accuracy: 0.6900
Epoch [13/20], Loss: 0.8400, Accuracy: 0.6955
Epoch [14/20], Loss: 0.8304, Accuracy: 0.7030
Epoch [15/20], Loss: 0.8280, Accuracy: 0.6986
Epoch [16/20], Loss: 0.8080, Accuracy: 0.7095
Epoch [17/20], Loss: 0.7973, Accuracy: 0.7109
Epoch [18/20], Loss: 0.7792, Accuracy: 0.7200
Epoch [19/20], Loss: 0.7644, Accuracy: 0.7265
Epoch [20/20], Loss: 0.7651, Accuracy: 0.7235
Finished Training. Total time: 1960.46 seconds

Evalua

### 9. Generate and Save Comprehensive Procedure Report

To keep a complete record of our work, we'll generate a text file containing all the markdown explanations and code snippets from this notebook. This report will be saved to your Google Drive, providing a detailed overview of the data preparation, model definition, training, and evaluation steps.

In [35]:
import os
from IPython import get_ipython
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np # Needed for array conversion for confusion matrix printing

# Define the path to save the comprehensive report
report_save_path = os.path.join(save_dir, 'colab_session_report.txt')

print(f"Generating comprehensive report and saving to: {report_save_path}")

report_content = []
report_content.append("### Colab Session Report: Emotion Recognition Model Training ###\n\n")
report_content.append("This report details the steps taken during the Colab session for training an emotion recognition model.\n\n")

# Get the IPython shell object
ipython = get_ipython()

if ipython is not None:
    # Add a section for Data Extraction and Initial Setup
    report_content.append(f"""## 1. Data Extraction and Initial Setup
The 'New_dataset.zip' file was extracted from Google Drive to './extracted_dataset'.
Code (cell 26999912):
```python
import os

# 1. Define the full path to New_dataset.zip in Google Drive
zip_file_path = '/content/drive/MyDrive/New_dataset.zip'

# 2. Create a new local directory for extraction
extraction_dir = './extracted_dataset'
if not os.path.exists(extraction_dir):
    os.makedirs(extraction_dir)
    print(f"Created extraction directory: {{extraction_dir}}")
else:
    print(f"Extraction directory already exists: {{extraction_dir}}")

# 3. Use the unzip command to extract the contents
#    -o flag for overwrite without prompt
#    -d flag to specify the destination directory
print(f"\\nExtracting '{{zip_file_path}}' to '{{extraction_dir}}'...")
!unzip -o "{{zip_file_path}}" -d "{{extraction_dir}}"

print("\\nExtraction complete. Verifying contents:")
# 4. After extraction, list the contents of the extracted_dataset directory recursively
!ls -R "{{extraction_dir}}"
```

""")

    # Add a section for Initial Data Inspection
    report_content.append(f"""## 2. Initial Data Inspection
Contents of the extracted dataset were listed to identify label files and image directories.
Code (cell 4e9d302c):
```python
print("Contents of 'extracted_dataset':")
!ls -F extracted_dataset

print("\\nAttempting to list 'extracted_dataset/DATASET':")
!ls -F extracted_dataset/DATASET
```

""")

    # Add a section for Loading Labels and Initial Mapping
    report_content.append(f"""## 3. Loading Labels and Initial Mapping
Train and test labels were loaded from CSV files into pandas DataFrames. Numerical labels were mapped to emotion names.
Code (cell 92caaf3a):
```python
import pandas as pd

train_labels_path = 'extracted_dataset/train_labels.csv'
test_labels_path = 'extracted_dataset/test_labels.csv'

train_df = pd.read_csv(train_labels_path)
test_df = pd.read_csv(test_labels_path)

print("Train Labels DataFrame Head:")
print(train_df.head())

print("\\nTest Labels DataFrame Head:")
print(test_df.head())
```
Code (cell 8c2b9a60):
```python
emotion_map = {{
    1: 'Surprise',
    2: 'Fear',
    3: 'Disgust',
    4: 'Happiness',
    5: 'Sadness',
    6: 'Anger',
    7: 'Neutral'
}}

train_df['emotion'] = train_df['label'].map(emotion_map)
test_df['emotion'] = test_df['label'].map(emotion_map)

print("\\nTrain Labels DataFrame with Emotion Names:")
print(train_df.head())

print("\\nTest Labels DataFrame with Emotion Names:")
print(test_df.head())
```

""")

    # Add a section for Constructing Full Image Paths
    report_content.append(f"""## 4. Constructing Full Image Paths
Absolute paths to images were constructed by incorporating the numerical label as a subdirectory.
Code (cell 0dcbb395):
```python
import os
base_image_dir = 'extracted_dataset/DATASET'
train_df['path'] = base_image_dir + '/train/' + train_df['label'].astype(str) + '/' + train_df['image']
test_df['path'] = base_image_dir + '/test/' + test_df['label'].astype(str) + '/' + test_df['image']
print("Train DataFrame with full image paths:")
print(train_df.head())
print("\\nTest DataFrame with full image paths:")
print(test_df.head())
```

""")

    # Add a section for Verifying Image Existence
    report_content.append(f"""## 5. Verifying Image Existence
Image paths were verified, and non-existent entries were removed to ensure data integrity.
Code (cell ce6224eb):
```python
def check_file_exists(file_path):
    return os.path.exists(file_path)

train_df['file_exists'] = train_df['path'].apply(check_file_exists)
test_df['file_exists'] = test_df['path'].apply(check_file_exists)

train_df_cleaned = train_df[train_df['file_exists']].drop(columns=['file_exists'])
test_df_cleaned = test_df[test_df['file_exists']].drop(columns=['file_exists'])

train_df = train_df_cleaned
test_df = test_df_cleaned

print(f"Original train_df size: {{len(train_df_cleaned)}}, Cleaned train_df size: {{len(train_df)}}")
print(f"Original test_df size: {{len(test_df_cleaned)}}, Cleaned test_df size: {{len(test_df)}}")
```

""")

    # Add a section for Encoding Categorical Labels
    report_content.append(f"""## 6. Encoding Categorical Labels
Emotion names were converted to numerical labels using `LabelEncoder` for model compatibility.
Code (cell df8bb7de):
```python
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
all_emotions = pd.concat([train_df['emotion'], test_df['emotion']], axis=0).unique()
label_encoder.fit(all_emotions)
train_df['encoded_label'] = label_encoder.transform(train_df['emotion'])
test_df['encoded_label'] = label_encoder.transform(test_df['emotion'])
print("Emotion to Encoded Label Mapping:")
for i, emotion in enumerate(label_encoder.classes_):
    print(f"{{emotion}}: {{i}}")
```

""")

    # Add a section for Image Preprocessing and Data Loaders
    report_content.append(f"""## 7. Image Preprocessing and Data Loaders
Custom `EmotionDataset` and `DataLoader` instances were created with defined image transformations (resizing, normalization, augmentation for training).
Code (cells 802b037a, 8c2aec64, 97f3b88b):
```python
import torch
from torchvision import transforms
from PIL import Image
from torch.utils.data import Dataset, DataLoader

IMAGE_SIZE = 224
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
test_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class EmotionDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_path = self.dataframe.iloc[idx]['path']
        image = Image.open(img_path).convert('RGB')
        label = self.dataframe.iloc[idx]['encoded_label']

        if self.transform:
            image = self.transform(image)

        return image, label

BATCH_SIZE = 32
train_dataset = EmotionDataset(dataframe=train_df, transform=train_transforms)
test_dataset = EmotionDataset(dataframe=test_df, transform=test_transforms)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
```

""")

    # Add a section for Model Architecture, Loss Function, and Optimizer
    report_content.append(f"""## 8. Model Architecture, Loss Function, and Optimizer
A pre-trained ResNet18 model was loaded using transfer learning, with its final layer adapted for 7 emotion classes. Cross-Entropy Loss and Adam optimizer were defined.
Code (cell fae9175d):
```python
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {{device}}")

num_classes = len(label_encoder.classes_)
print(f"Number of emotion classes: {{num_classes}}")

model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, num_classes)
model = model.to(device)

print("Pre-trained ResNet18 model defined and instantiated.")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("Loss function (CrossEntropyLoss) and optimizer (Adam) defined.")
```

""")

    # Add a section for Model Training and Evaluation
    report_content.append(f"""## 9. Model Training and Evaluation
The model was trained for {NUM_EPOCHS} epochs, and its performance was evaluated on the test set. Training and evaluation functions were defined.
Code (cells b40fb1e4, 3944c462):
```python
import time
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np # Added for confusion matrix formatting

def train_model(model, train_loader, criterion, optimizer, num_epochs=10):
    model.train() # Set the model to training mode
    start_time = time.time()

    for epoch in range(num_epochs):
        running_loss = 0.0
        correct_predictions = 0
        total_samples = 0

        for i, (inputs, labels) in enumerate(train_loader):
            inputs, labels = inputs.to(device), labels.to(device)

            # Zero the parameter gradients
            optimizer.zero_grad()

            # Forward pass
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            # Backward pass and optimize
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)

            _, predicted = torch.max(outputs.data, 1)
            total_samples += labels.size(0)
            correct_predictions += (predicted == labels).sum().item()

        epoch_loss = running_loss / total_samples
        epoch_accuracy = correct_predictions / total_samples
        print(f'Epoch [{{epoch+1}}/{{num_epochs}}], Loss: {{epoch_loss:.4f}}, Accuracy: {{epoch_accuracy:.4f}}')

    end_time = time.time()
    print(f'Finished Training. Total time: {{end_time - start_time:.2f}} seconds')

# ### 7. Model Evaluation
#
# After training, we need to evaluate the model's performance on the unseen test dataset.

def evaluate_model(model, test_loader):
    model.eval() # Set the model to evaluation mode
    correct_predictions = 0
    total_samples = 0
    all_predictions = []
    all_true_labels = []

    with torch.no_grad(): # Disable gradient calculation during evaluation
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)

            total_samples += labels.size(0)
            correct_predictions += (predicted == labels).sum().item()

            all_predictions.extend(predicted.cpu().numpy())
            all_true_labels.extend(labels.cpu().numpy())

    accuracy = correct_predictions / total_samples
    print(f'Accuracy on the test set: {{accuracy:.4f}}')
    return accuracy, all_predictions, all_true_labels

NUM_EPOCHS = {NUM_EPOCHS} # This will be replaced by the actual value of NUM_EPOCHS from the context
train_model(model, train_loader, criterion, optimizer, num_epochs=NUM_EPOCHS)
test_accuracy, all_predictions, all_true_labels = evaluate_model(model, test_loader)

save_dir = '/content/drive/MyDrive/emotion_model_results'
os.makedirs(save_dir, exist_ok=True)
torch.save(model.state_dict(), os.path.join(save_dir, 'resnet18_emotion_model.pth'))
with open(os.path.join(save_dir, 'evaluation_report.txt'), 'w') as f:
    f.write(f"Final Test Accuracy: {{test_accuracy:.4f}}\\n\\n")

    # Generate and write classification report
    target_names = [label_encoder.inverse_transform([i])[0] for i in range(num_classes)]
    report = classification_report(all_true_labels, all_predictions, target_names=target_names)
    f.write("Classification Report:\\n")
    f.write(report)
    f.write("\\n\\n")

    # Generate and write confusion matrix
    cm = confusion_matrix(all_true_labels, all_predictions)
    f.write("Confusion Matrix:\\n")
    # Convert confusion matrix to a string format for better readability in the report
    cm_str = np.array2string(cm, separator=', ')
    f.write(cm_str)
    f.write("\\n")
```

""")

    # Final summary
    report_content.append(f"""## Conclusion
The model achieved a final test accuracy of {test_accuracy:.4f} after training for {NUM_EPOCHS} epochs, demonstrating successful transfer learning for emotion recognition.
The trained model and evaluation metrics have been saved to '{save_dir}'.
""")

    # Write the compiled report to the file
    with open(report_save_path, 'w') as f:
        f.writelines(report_content)
    print("Comprehensive report successfully generated and saved.")
else:
    print("Could not get IPython shell to generate comprehensive report.")

Generating comprehensive report and saving to: /content/drive/MyDrive/emotion_model_results/colab_session_report.txt
Comprehensive report successfully generated and saved.


### Create Dataset Instances and Data Loaders

Now, we'll instantiate our `EmotionDataset` for both training and testing, and then create `DataLoader` instances to efficiently iterate over the datasets during model training.

In [30]:
# Create dataset instances
train_dataset = EmotionDataset(dataframe=train_df, transform=train_transforms)
test_dataset = EmotionDataset(dataframe=test_df, transform=test_transforms)

# Define batch size
BATCH_SIZE = 32 # You can adjust this based on your GPU memory

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Training dataset size: {len(train_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")
print(f"Training data loader created with batch size: {BATCH_SIZE}")
print(f"Test data loader created with batch size: {BATCH_SIZE}")

# You can optionally check one batch to ensure everything is working
# for images, labels in train_loader:
#     print(f"Batch image shape: {images.shape}") # Expected: [BATCH_SIZE, 3, IMAGE_SIZE, IMAGE_SIZE]
#     print(f"Batch labels shape: {labels.shape}") # Expected: [BATCH_SIZE]
#     break

Training dataset size: 12271
Test dataset size: 3068
Training data loader created with batch size: 32
Test data loader created with batch size: 32


### 5. Define Model Architecture, Loss Function, and Optimizer

Now we'll define the neural network model. For image classification, a Convolutional Neural Network (CNN) is a standard choice. We'll also specify the loss function and the optimizer.

In [29]:
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models # Import torchvision models

# Check if GPU is available and set the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Get the number of unique classes from the label_encoder
num_classes = len(label_encoder.classes_)
print(f"Number of emotion classes: {num_classes}")

# Define the model architecture using a pre-trained ResNet
# We'll use ResNet18 for a good balance of performance and speed
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1) # Load pre-trained ResNet18

# Freeze all parameters in the network
# for param in model.parameters():
#     param.requires_grad = False

# Replace the final fully connected layer (classifier)
# The number of input features to the fc layer depends on the chosen ResNet variant
# For ResNet18, it's 512
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, num_classes) # New layer with num_classes outputs

# Move the model to the appropriate device (GPU or CPU)
model = model.to(device)

print("Pre-trained ResNet18 model defined and instantiated.")

# Define Loss Function and Optimizer
criterion = nn.CrossEntropyLoss() # Suitable for multi-class classification

# Only optimize parameters that are not frozen (if we froze them)
# If we don't freeze, all parameters will be optimized
optimizer = optim.Adam(model.parameters(), lr=0.001) # Adam optimizer with a learning rate of 0.001

print("Loss function (CrossEntropyLoss) and optimizer (Adam) defined.")

Using device: cuda
Number of emotion classes: 7
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 162MB/s]


Pre-trained ResNet18 model defined and instantiated.
Loss function (CrossEntropyLoss) and optimizer (Adam) defined.
